In [1]:
!pip -q install openai datasets pandas tqdm tenacity

In [2]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
print("API key loaded securely.")

Enter your OpenAI API key: ··········
API key loaded securely.


In [3]:
from datasets import load_dataset

dataset = load_dataset("gtfintechlab/finer-ord-bio")

print(dataset)
print(dataset["test"].column_names)
print(dataset["test"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


data/train-00000-of-00001.parquet:   0%|          | 0.00/279k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/41.6k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/93.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3262 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/402 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1075 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 3262
    })
    validation: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 402
    })
    test: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 1075
    })
})
['tokens', 'tags']
{'tokens': ['H', 'ave', 'you', 'ever', 'felt', 'that', 'sudden', ',', 'intense', 'dread', 'that', 'you', '’re', 'about', 'to', 'die', '?'], 'tags': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


In [8]:
import os
import json
import time
import pandas as pd
from tqdm import tqdm
from collections import Counter
from tenacity import retry, wait_exponential, stop_after_attempt

from datasets import load_dataset
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

MODEL = "gpt-5.5"

# First test with 3 or 5.
# Then set LIMIT = None to evaluate the full test set.
LIMIT = 100

# Use the accessible FiNER-ORD BIO dataset.
DATASET_NAME = "gtfintechlab/finer-ord-bio"
dataset = load_dataset(DATASET_NAME)

VALID_LABELS = [
    "O",
    "B-PER", "I-PER",
    "B-LOC", "I-LOC",
    "B-ORG", "I-ORG"
]

LABEL_SCHEMA = {
    "type": "object",
    "properties": {
        "labels": {
            "type": "array",
            "items": {
                "type": "string",
                "enum": VALID_LABELS
            }
        }
    },
    "required": ["labels"],
    "additionalProperties": False
}

SYSTEM_PROMPT = """
You are a financial named entity recognition model.

Task:
Given a list of tokens from a financial text, assign one BIO label to each token.

Allowed labels:
- O
- B-PER, I-PER
- B-LOC, I-LOC
- B-ORG, I-ORG

Rules:
1. Return exactly one label for each input token.
2. The number of output labels must equal the number of input tokens.
3. Use BIO format.
4. Do not add explanations.
5. Return only valid JSON.
"""

@retry(wait=wait_exponential(multiplier=1, min=2, max=30), stop=stop_after_attempt(5))
def call_openai_token_ner(tokens):
    indexed_tokens = "\n".join([f"{i}: {tok}" for i, tok in enumerate(tokens)])

    response = client.responses.create(
        model=MODEL,
        input=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": (
                    f"Number of tokens: {len(tokens)}\n\n"
                    f"Tokens:\n{indexed_tokens}\n\n"
                    f"Return exactly {len(tokens)} BIO labels."
                )
            }
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": "finer_ord_ner_labels",
                "schema": LABEL_SCHEMA,
                "strict": True
            }
        },
        reasoning={
            "effort": "low"
        },
        max_output_tokens=1200
    )

    return json.loads(response.output_text)

def normalize_label(label):
    label = str(label).strip()

    # Convert FiNER style to BIO style if needed.
    # PER_B -> B-PER
    # PER_I -> I-PER
    if label in ["PER_B", "LOC_B", "ORG_B"]:
        return "B-" + label.split("_")[0]

    if label in ["PER_I", "LOC_I", "ORG_I"]:
        return "I-" + label.split("_")[0]

    if label in VALID_LABELS:
        return label

    return "O"

def get_column_name(example, possible_names):
    for name in possible_names:
        if name in example:
            return name

    raise ValueError(
        f"Cannot find any column from {possible_names}. "
        f"Available columns are: {list(example.keys())}"
    )

def int_label_to_string(label_id, split_data, label_col):
    feature = split_data.features[label_col]

    # Case 1: Sequence(ClassLabel)
    if hasattr(feature, "feature") and hasattr(feature.feature, "int2str"):
        return feature.feature.int2str(label_id)

    # Case 2: ClassLabel
    if hasattr(feature, "int2str"):
        return feature.int2str(label_id)

    return str(label_id)

def convert_labels(raw_labels, split_data, label_col):
    labels = []

    for x in raw_labels:
        if isinstance(x, int):
            label_name = int_label_to_string(x, split_data, label_col)
            labels.append(normalize_label(label_name))
        else:
            labels.append(normalize_label(x))

    return labels

def fix_prediction_length(pred_labels, target_len):
    pred_labels = [normalize_label(x) for x in pred_labels]

    if len(pred_labels) < target_len:
        pred_labels = pred_labels + ["O"] * (target_len - len(pred_labels))

    if len(pred_labels) > target_len:
        pred_labels = pred_labels[:target_len]

    return pred_labels

def bio_to_entities(labels):
    entities = []
    start = None
    ent_type = None

    for i, label in enumerate(labels + ["O"]):
        if label == "O":
            if start is not None:
                entities.append((start, i, ent_type))
                start = None
                ent_type = None
            continue

        if "-" not in label:
            continue

        prefix, label_type = label.split("-", 1)

        if prefix == "B":
            if start is not None:
                entities.append((start, i, ent_type))
            start = i
            ent_type = label_type

        elif prefix == "I":
            if start is None:
                start = i
                ent_type = label_type
            elif ent_type != label_type:
                entities.append((start, i, ent_type))
                start = i
                ent_type = label_type

    return entities

def compute_metrics(gold_all, pred_all):
    correct_tokens = 0
    total_tokens = 0
    exact_match_count = 0

    tp = 0
    fp = 0
    fn = 0

    for gold_labels, pred_labels in zip(gold_all, pred_all):
        total_tokens += len(gold_labels)
        correct_tokens += sum(g == p for g, p in zip(gold_labels, pred_labels))

        if gold_labels == pred_labels:
            exact_match_count += 1

        gold_entities = Counter(bio_to_entities(gold_labels))
        pred_entities = Counter(bio_to_entities(pred_labels))

        tp += sum((gold_entities & pred_entities).values())
        fp += sum((pred_entities - gold_entities).values())
        fn += sum((gold_entities - pred_entities).values())

    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    token_accuracy = correct_tokens / total_tokens if total_tokens > 0 else 0.0
    exact_match_accuracy = exact_match_count / len(gold_all) if len(gold_all) > 0 else 0.0

    return {
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "token_accuracy": token_accuracy,
        "exact_match_accuracy": exact_match_accuracy,
        "true_positive": tp,
        "false_positive": fp,
        "false_negative": fn
    }

# Choose test split if available.
split_name = "test" if "test" in dataset else list(dataset.keys())[0]
split_data = dataset[split_name]

example = split_data[0]

print("Dataset:", DATASET_NAME)
print("Split:", split_name)
print("Available columns:", list(example.keys()))

token_col = get_column_name(example, ["tokens", "token", "gold_token"])
label_col = get_column_name(example, ["tags", "tag", "labels", "label", "gold_label"])

print("Token column:", token_col)
print("Label column:", label_col)

eval_data = split_data if LIMIT is None else split_data.select(range(LIMIT))

rows = []
gold_all = []
pred_all = []

for idx, row in enumerate(tqdm(eval_data)):
    tokens = row[token_col]
    gold_labels = convert_labels(row[label_col], split_data, label_col)

    try:
        result = call_openai_token_ner(tokens)
        pred_labels = result.get("labels", [])
        pred_labels = fix_prediction_length(pred_labels, len(gold_labels))
        error = ""
    except Exception as e:
        pred_labels = ["O"] * len(gold_labels)
        error = str(e)
        print(f"Error at row {idx}: {e}")

    gold_all.append(gold_labels)
    pred_all.append(pred_labels)

    rows.append({
        "idx": idx,
        "tokens": tokens,
        "gold_labels": gold_labels,
        "pred_labels": pred_labels,
        "gold_entities": bio_to_entities(gold_labels),
        "pred_entities": bio_to_entities(pred_labels),
        "exact_match": gold_labels == pred_labels,
        "error": error
    })

    time.sleep(10)

metrics = compute_metrics(gold_all, pred_all)

print("\n===== GPT-5.5 FiNER-ORD Evaluation =====")
print(f"Dataset: {DATASET_NAME}")
print(f"Split: {split_name}")
print(f"Model: {MODEL}")
print(f"Samples evaluated: {len(eval_data)}")
print(f"Precision / Correctness Rate: {metrics['precision']:.4f}")
print(f"Recall: {metrics['recall']:.4f}")
print(f"F1 Score: {metrics['f1_score']:.4f}")
print(f"Exact Match Accuracy: {metrics['exact_match_accuracy']:.4f}")
print(f"Token Accuracy: {metrics['token_accuracy']:.4f}")
print(f"TP: {metrics['true_positive']}")
print(f"FP: {metrics['false_positive']}")
print(f"FN: {metrics['false_negative']}")

df = pd.DataFrame(rows)
df.to_csv("gpt55_finer_ord_bio_evaluation_results.csv", index=False)

pd.set_option("display.max_colwidth", 120)
df.head()

Repo card metadata block was not found. Setting CardData to empty.


Dataset: gtfintechlab/finer-ord-bio
Split: test
Available columns: ['tokens', 'tags']
Token column: tokens
Label column: tags


100%|██████████| 100/100 [20:04<00:00, 12.04s/it]


===== GPT-5.5 FiNER-ORD Evaluation =====
Dataset: gtfintechlab/finer-ord-bio
Split: test
Model: gpt-5.5
Samples evaluated: 100
Precision / Correctness Rate: 0.0000
Recall: 0.0000
F1 Score: 0.0000
Exact Match Accuracy: 0.4800
Token Accuracy: 0.9342
TP: 0
FP: 113
FN: 0


,idx,tokens,gold_labels,pred_labels,gold_entities,pred_entities,exact_match,error
0,0,"[H, ave, you, ever, felt, that, sudden, ,, intense, dread, that, you, ’re, about, to, die, ?]","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]",[],[],True,
1,1,"[That, in, less, than, seven, minutes, ,, the, knot, of, fear, and, anxiety, unspooling, at, speed, will, swallow, y...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]",[],[],True,
2,2,"[And, all, because, you, failed, to, prepare, !]","[O, O, O, O, O, O, O, O]","[O, O, O, O, O, O, O, O]",[],[],True,
3,3,"[You, idiot, !]","[O, O, O]","[O, O, O]",[],[],True,
4,4,"[And, now, you, ’re, on, a, train, carriage, in, a, clammy, panic, desperately, searching, for, hope, .]","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]",[],[],True,
